# 02 — Time-Series Manipulation & Temporal Feature Engineering

This notebook explores time-series data handling, frequency alignment, rolling window aggregations, causal lag feature engineering, and avoiding lookahead bias in temporal machine learning pipelines.

## 1. Temporal Monotonicity and Causality

A time-series dataset is an indexed sequence of observations recorded across successive temporal points:
$$\mathcal{D} = \{(t_1, y_1), (t_2, y_2), \dots, (t_T, y_T)\}, \quad t_1 < t_2 < \dots < t_T$$

**The Cardinal Rule of Time-Series Engineering:**
At any time step $t$, feature calculations must rely strictly on historical information available up to time $t-1$ or $t$. Incorporating future observations $t+k$ ($k > 0$) introduces **temporal lookahead bias** (data leakage), yielding artificially optimistic validation scores that collapse in live deployment.

In [ ]:
import csv
import datetime
import math
from pathlib import Path

DATA_PATH = Path("../data/server_telemetry_timeseries.csv")
assert DATA_PATH.is_file(), f"Time-series dataset missing at {DATA_PATH}"
print(f"Verified time-series dataset: {DATA_PATH.resolve()}")

## 2. Ingesting and Parsing Temporal Telemetry

We parse the hourly telemetry records into structured Python dictionaries with explicit `datetime` objects.

In [ ]:
def load_timeseries(file_path: Path) -> list[dict]:
    records = []
    with file_path.open(encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = {
                "timestamp": datetime.datetime.fromisoformat(row["timestamp"]),
                "cpu_utilization": float(row["cpu_utilization_pct"]),
                "memory_utilization": float(row["memory_utilization_pct"]),
                "request_rate": float(row["request_rate_per_sec"]),
                "error_count": int(row["error_count"]),
                "p95_latency": float(row["p95_latency_ms"]),
            }
            records.append(parsed)
    return records


telemetry_data = load_timeseries(DATA_PATH)
print(f"Total time steps: {len(telemetry_data)}")
print(f"Earliest observation: {telemetry_data[0]['timestamp']}")
print(f"Latest observation:   {telemetry_data[-1]['timestamp']}")

## 3. Resampling and Temporal Aggregation

Downsampling aggregates high-frequency observations into coarser temporal intervals (e.g., hourly to daily). We aggregate 24 hourly observations into daily summary metrics (daily mean, daily peak max, daily total).

In [ ]:
def resample_daily(records: list[dict]) -> list[dict]:
    """Downsample hourly records to daily aggregates."""
    daily_bins: dict[datetime.date, list[dict]] = {}
    for r in records:
        day = r["timestamp"].date()
        daily_bins.setdefault(day, []).append(r)

    daily_summary = []
    for day, items in sorted(daily_bins.items()):
        cpus = [item["cpu_utilization"] for item in items]
        reqs = [item["request_rate"] for item in items]
        errs = [item["error_count"] for item in items]

        daily_summary.append(
            {
                "date": day,
                "cpu_mean": sum(cpus) / len(cpus),
                "cpu_max": max(cpus),
                "request_mean": sum(reqs) / len(reqs),
                "total_errors": sum(errs),
                "hours_count": len(items),
            }
        )
    return daily_summary


daily_telemetry = resample_daily(telemetry_data)
print(f"Aggregated into {len(daily_telemetry)} daily records. First 5 days:")
for d in daily_telemetry[:5]:
    print(
        f"  Date: {d['date']} | Mean CPU: {d['cpu_mean']:.2f}% | Peak CPU: {d['cpu_max']:.2f}% | Total Errors: {d['total_errors']}"
    )

## 4. Rolling Window Statistics and Moving Averages

Rolling statistics capture dynamic trends across a finite sliding historical horizon of length $w$:
- **Simple Moving Average (SMA):**
  $$	ext{SMA}_t(w) = rac{1}{w} \sum_{i=0}^{w-1} y_{t-i}$$
- **Exponential Moving Average (EMA):**
  $$	ext{EMA}_t(lpha) = lpha y_t + (1 - lpha) 	ext{EMA}_{t-1}, \quad lpha = rac{2}{w + 1}$$
- **Rolling Volatility (Standard Deviation):**
  $$\sigma_t(w) = \sqrt{rac{1}{w-1} \sum_{i=0}^{w-1} (y_{t-i} - 	ext{SMA}_t)^2}$$

In [ ]:
def compute_rolling_statistics(
    series: list[float], window_size: int
) -> tuple[list[float | None], list[float | None]]:
    """Compute causal rolling mean and rolling standard deviation."""
    rolling_means: list[float | None] = []
    rolling_stds: list[float | None] = []

    for i in range(len(series)):
        if i + 1 < window_size:
            rolling_means.append(None)
            rolling_stds.append(None)
            continue

        window = series[i + 1 - window_size : i + 1]
        mean_val = sum(window) / window_size
        variance = sum((x - mean_val) ** 2 for x in window) / (window_size - 1)
        rolling_means.append(mean_val)
        rolling_stds.append(math.sqrt(variance))

    return rolling_means, rolling_stds


cpu_series = [r["cpu_utilization"] for r in telemetry_data]
sma_24, std_24 = compute_rolling_statistics(cpu_series, window_size=24)

print("Sample 24-hour Rolling CPU Statistics:")
for i in [23, 24, 48, 72, 100]:
    print(
        f"  Hour {i}: Actual CPU = {cpu_series[i]:.2f}%, 24h SMA = {sma_24[i]:.2f}%, 24h Std = {std_24[i]:.2f}%"
    )

## 5. Causal Temporal Feature Engineering

To train forecasting or anomaly detection models, we construct:
1. **Lags:** $y_{t-1}, y_{t-2}, y_{t-24}$
2. **First Difference:** $\Delta y_t = y_t - y_{t-1}$
3. **Cyclical Calendar Features:** Projecting hour $h \in [0, 23]$ onto the unit circle:
   $$h_{\sin} = \sin\left(rac{2\pi h}{24}ight), \quad h_{\cos} = \cos\left(rac{2\pi h}{24}ight)$$

In [ ]:
def engineer_temporal_features(records: list[dict]) -> list[dict]:
    features = []
    for i, r in enumerate(records):
        t = r["timestamp"]
        hour = t.hour
        h_sin = math.sin(2.0 * math.pi * hour / 24.0)
        h_cos = math.cos(2.0 * math.pi * hour / 24.0)

        # Lags
        lag_1 = records[i - 1]["cpu_utilization"] if i >= 1 else None
        lag_24 = records[i - 24]["cpu_utilization"] if i >= 24 else None
        diff_1 = (r["cpu_utilization"] - lag_1) if lag_1 is not None else None

        features.append(
            {
                "timestamp": t.isoformat(),
                "hour_sin": round(h_sin, 4),
                "hour_cos": round(h_cos, 4),
                "cpu_actual": r["cpu_utilization"],
                "cpu_lag_1": lag_1,
                "cpu_lag_24": lag_24,
                "cpu_diff_1": round(diff_1, 2) if diff_1 is not None else None,
            }
        )
    return features


feature_matrix = engineer_temporal_features(telemetry_data)
print("Engineered temporal feature rows (Hours 24-26):")
for f_row in feature_matrix[24:27]:
    print(" ", f_row)